# e_simulate

Run all cells. Outputs are written to this task's `output/` folder.


In [1]:
%run ~/Desktop/SHL_dblp_comparable/common/core.ipynb


/Users/slmagid/miniforge3/envs/research313/lib/python3.13/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [2]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
ROOT = Path.home() / 'Desktop' / 'SHL_dblp_comparable'
CONFIG = read_config(ROOT)
UP = ROOT / 'd_fit' / 'output'
PREP = ROOT / 'c_prepare' / 'output'
OUTPUT = ROOT / 'e_simulate' / 'output'
TRAJ = OUTPUT / 'trajectories'
RESULTS = OUTPUT / 'results'
for p in (TRAJ, RESULTS):
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)
n_sim = int(CONFIG['n_simulations'])
lengths = pd.read_csv(PREP / 'trajectory_lengths.csv')
summary = []
for path in sorted((UP / 'models').glob('*.pkl')):
    bundle = load_bundle(path)
    group = bundle['analysis_group']
    observed = lengths[lengths.analysis_group.eq(group) & lengths.split.eq('test')].observed_years.to_numpy(int)
    for name in (MODEL_T5, MODEL_AR):
        q, l = simulate(bundle['models'][name], name, bundle['rho_t5'], n_sim, 12, stable_seed(group, name, 'simulation'), observed)
        filename = f"{path.stem}__{('t5' if name == MODEL_T5 else 'unrestricted-ar6')}.npz"
        np.savez_compressed(TRAJ / filename, q=q, lengths=l, analysis_group=group, model=name)
        summary.append({'analysis_group': group, 'model': name, 'simulations': n_sim, 'mean_observed_years': float(l.mean()), 'file': filename})
pd.DataFrame(summary).to_csv(RESULTS / 'simulation_summary.csv', index=False)
write_json(OUTPUT / 'quality_report.json', {'status': 'passed', 'n_simulations': n_sim, 'observation_mask': 'held-out cohort-specific lengths'})
print(pd.DataFrame(summary).to_string(index=False))


              analysis_group                       model  simulations  mean_observed_years                                               file
      all_eligible_cropped13                SE-Hurdle-T5        10000              10.7245                     all-eligible-cropped13__t5.npz
      all_eligible_cropped13 Unrestricted-Hurdle-AR(6,6)        10000              10.6989       all-eligible-cropped13__unrestricted-ar6.npz
all_eligible_no_aarc_overlap                SE-Hurdle-T5        10000              10.7357               all-eligible-no-aarc-overlap__t5.npz
all_eligible_no_aarc_overlap Unrestricted-Hurdle-AR(6,6)        10000              10.7368 all-eligible-no-aarc-overlap__unrestricted-ar6.npz
          complete13_cropped                SE-Hurdle-T5        10000              13.0000                         complete13-cropped__t5.npz
          complete13_cropped Unrestricted-Hurdle-AR(6,6)        10000              13.0000           complete13-cropped__unrestricted-ar6.npz
 legac